In [0]:
# Date parameter
dbutils.widgets.text("current_date", "")
current_date = dbutils.widgets.get("current_date")
print(current_date)

# Configuration Define file path and Stage table 
# (here f" is Important -> means formatted string literal (also called an f-string) in Python.It allows you to insert variables directly inside a string using {}.)

source_dir = f"/Volumes/zoomcarprocessingpipeline/default/processing_data/bookings/source/zoom_car_bookings_{current_date}.json"
archive_dir = f"/Volumes/zoomcarprocessingpipeline/default/processing_data/bookings/archive/zoom_car_bookings_{current_date}.json"

stage_table = "`zoomcarprocessingpipeline`.default.bookings_stage"
error_table = "`zoomcarprocessingpipeline`.default.bookings_errors"

print(f"Processing orders data from: {source_dir}")
print(f"Staging table: {stage_table}")

In [0]:
# Import required libraries
from pyspark.sql import functions as F
from pyspark.sql.types import *
from datetime import datetime
import json

# Define schema for booking data
booking_schema = StructType([
    StructField("booking_id", StringType(), True),
    StructField("customer_id", StringType(), True),
    StructField("car_id", StringType(), True),
    StructField("booking_date", DateType(), True),
    StructField("start_time", TimestampType(), True),
    StructField("end_time", TimestampType(), True),
    StructField("total_amount", DoubleType(), True),
    StructField("status", StringType(), True)
])

print("Schema defined for booking data")


rawbooking_df = (spark.read.schema(booking_schema).json(source_dir))

rawbooking_df.show()

In [0]:
# Check for null values in each column
null_counts = rawbooking_df.agg(
    *[F.sum(F.col(column).isNull().cast("int")).alias(f"{column}_null_count") for column in rawbooking_df.columns]
)

# Check for data types
data_type_checks = [F.col(column).cast("string").alias(f"{column}_type_check") for column in rawbooking_df.columns]

# Apply the data type checks
df_check = rawbooking_df.select(data_type_checks)

# Show the results of the checks
print("Null Counts:")
null_counts.show()

print("Data Type Checks:")
df_check.show()

In [0]:
valid_statuses = ["completed", "cancelled", "pending"]
# Remove nulls from critical fields
clean_df = (rawbooking_df.filter(
        F.col("booking_id").isNotNull() &
        F.col("customer_id").isNotNull() &
        F.col("car_id").isNotNull() &
        F.col("booking_date").isNotNull()
    )
)
clean_df = clean_df.withColumn("booking_date", F.to_date(F.col("booking_date"), "yyyy-MM-dd"))
clean_df = (clean_df.withColumn("start_time",F.to_timestamp("start_time")).withColumn("end_time",F.to_timestamp("end_time")))
clean_df = clean_df.filter(F.col("start_time").isNotNull() & F.col("end_time").isNotNull())
# Validate status values
clean_df = clean_df.filter(F.col("status").isin(valid_statuses))

# Remove duplicates
clean_df = clean_df.dropDuplicates(["booking_id"])

 # Filter out valid records - Fixed boolean logic
df_valid_booking = clean_df.filter(
        (F.col("booking_id").isNotNull()) & 
        (F.col("customer_id").isNotNull()) & 
        (F.col("total_amount") > 0)
    )
    
    # Capture invalid records for error handling - Fixed boolean logic
df_invalid_booking = clean_df.filter(
        (F.col("booking_id").isNull()) | 
        (F.col("customer_id").isNull()) | 
        (F.col("total_amount") <= 0)
    )
    
valid_records = df_valid_booking.count()
invalid_records = df_invalid_booking.count()

print(f"Valid records: {valid_records}")
print(f"Invalid records: {invalid_records}")

clean_df.show()

In [0]:
# Write valid data to staging table
try:
    # Create or overwrite staging table
    df_valid_booking.write.format("delta").mode("overwrite").saveAsTable(stage_table)
    print(f"Successfullyloaded {valid_records} valid booking to staging table")
    
    # Write invalid records to error table for investigation
    if invalid_records > 0:
        df_invalid_booking.write.withColumn("error_reason", F.lit("Data quality validation failed")) \
                           .withColumn("error_timestamp", F.current_timestamp()) \
                           .write.format("delta").mode("append").saveAsTable(error_table)
        print(f"Logged {invalid_records} invalid records to error table")
    
except Exception as e:
    print(f"Error writing to staging table: {str(e)}")
    raise

In [0]:
# Archive processed files
try:
    # List all files in the source directory
    files = dbutils.fs.ls(source_dir)
    
    archived_count = 0
    for file in files:
        if file.name.endswith('.json'):
            src_path = file.path
            archive_path = archive_dir + file.name
            
            # Move the file to archive
            dbutils.fs.mv(src_path, archive_path)
            archived_count += 1
            print(f"Archived: {file.name}")
    
    print(f"Successfully archived {archived_count} files")
    
except Exception as e:
    print(f"Error archiving files: {str(e)}")
    raise